In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
from datasets import load_dataset

In [ ]:
def create_masked_samples(dataset, tokenizer, n_samples=3000):
    samples = []

    for x in dataset:
        text = x["text"]

        words = text.split()

        for word in words:
            tokens = tokenizer.tokenize(word)

            # need at least 3 tokens to simulate morphology
            if len(tokens) < 3:
                continue

            # choose middle token (approx morphological core)
            idx = len(tokens) // 2
            target_token = tokens[idx]

            # mask that token
            masked_tokens = tokens.copy()
            masked_tokens[idx] = tokenizer.mask_token

            # reconstruct masked word
            masked_word = tokenizer.convert_tokens_to_string(masked_tokens)

            # replace ONLY ONE occurrence
            masked_sentence = text.replace(word, masked_word, 1)

            samples.append((masked_sentence, target_token))

            if len(samples) >= n_samples:
                return samples

    return samples

In [ ]:
from tqdm import tqdm
import torch

def evaluate_morphology(samples, tokenizer, model, top_k=5):
    correct = 0
    total = 0

    for sentence, target_token in tqdm(samples, desc="Evaluating morphology"):
        inputs = tokenizer(sentence, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        logits = outputs.logits
        input_ids = inputs["input_ids"][0]

        mask_token_id = tokenizer.mask_token_id

        for i, token_id in enumerate(input_ids):
            if token_id == mask_token_id:
                probs = logits[0, i].softmax(dim=-1)

                top_tokens = torch.topk(probs, k=top_k).indices
                decoded = tokenizer.convert_ids_to_tokens(top_tokens)
                decoded = [t.replace('Ġ', '').replace(' ', '') for t in decoded]
                clean_target = target_token.replace('Ġ', '').replace(' ', '')
                if clean_target in decoded:
                    correct += 1

                total += 1

    return correct / total if total > 0 else 0.0

# Load datasets

In [ ]:
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-103-raw-v1"

MAX_LENGTH = 128
BATCH_SIZE = 8
EVAL_SAMPLES = 3000

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
device

'cuda'

In [ ]:
dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split="test")

dataset = dataset.filter(lambda x: x["text"] and len(x["text"].strip()) > 0)
dataset_en = dataset.select(range(min(EVAL_SAMPLES, len(dataset))))

dataset_en

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4358 [00:00<?, ? examples/s]

Dataset({
    features: ['text'],
    num_rows: 2891
})

In [ ]:
dataset = load_dataset("ngusadeep/Swahili-Corpus-Dataset")["train"]

# Remove empty lines
dataset = dataset.filter(lambda x: len(x["text"].strip()) > 0)

# Limit dataset size (low-resource simulation)
dataset_dict = dataset.train_test_split(test_size=0.1, seed=42, shuffle=True)

raw_train_data = dataset_dict["train"]
eval_data = dataset_dict["test"]

dataset_swa = eval_data.select(range(min(3000, len(eval_data))))

print(f"Eval size:  {len(dataset_swa)}")

README.md: 0.00B [00:00, ?B/s]

Swahili_Corpus_combined.txt:   0%|          | 0.00/253M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1693227 [00:00<?, ? examples/s]

Eval size:  3000


# Eval roberta_en

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/english_lm_roberta"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
model.eval().to(device)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [ ]:
samples = create_masked_samples(dataset, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:49<00:00, 60.02it/s]

Morphological generalization score: 0.3393333333333333


In [ ]:
samples = create_masked_samples(dataset_swa, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:50<00:00, 59.48it/s]

Morphological generalization score: 0.323


# Eval roberta_swahili (continuous pretraining)

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/roberta_swahili"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
model.eval().to(device)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): 

In [ ]:
samples = create_masked_samples(dataset, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:44<00:00, 66.91it/s]

Morphological generalization score: 0.718


In [ ]:
samples = create_masked_samples(dataset_swa, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:55<00:00, 54.45it/s]

Morphological generalization score: 0.6656666666666666


# Eval roberta_swahili_retokenize

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch

MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/roberta_swahili_retokenize"
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
model.eval().to(device)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

RobertaForMaskedLM(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(6109, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): L

In [ ]:
samples = create_masked_samples(dataset, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:38<00:00, 76.96it/s]

Morphological generalization score: 0.105


In [ ]:
samples = create_masked_samples(dataset_swa, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:35<00:00, 83.39it/s]

Morphological generalization score: 0.101


# Eval roberta_swahili_adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM
import torch
from peft import PeftModel

MODEL_PATH = "/content/drive/MyDrive/nlp_project/models/roberta_swahili_retokenize"
ADAPTER_PATH = "/content/drive/MyDrive/nlp_project/models/roberta_swahili_adapter"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
base_model = AutoModelForMaskedLM.from_pretrained(MODEL_PATH)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval().to(device)


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

PeftModelForFeatureExtraction(
  (base_model): LoraModel(
    (model): RobertaForMaskedLM(
      (roberta): RobertaModel(
        (embeddings): RobertaEmbeddings(
          (word_embeddings): Embedding(6109, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (position_embeddings): Embedding(514, 768, padding_idx=1)
        )
        (encoder): RobertaEncoder(
          (layer): ModuleList(
            (0-11): 12 x RobertaLayer(
              (attention): RobertaAttention(
                (self): RobertaSelfAttention(
                  (query): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.1, inplace=False)
                    )
                    (lora_A): ModuleDict(
               

In [ ]:
samples = create_masked_samples(dataset, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:51<00:00, 58.31it/s]

Morphological generalization score: 0.138


In [ ]:
samples = create_masked_samples(dataset_swa, tokenizer, n_samples=3000)
score = evaluate_morphology(samples, tokenizer, model)

print("Morphological generalization score:", score)

Evaluating morphology: 100%|██████████| 3000/3000 [00:50<00:00, 59.63it/s]

Morphological generalization score: 0.10633333333333334
